# 👟 ARShoe M1v2 13-Keypoint Pipeline: Raw Dataset Prep, Training & Evaluation

This notebook runs the complete 13-keypoint ablation study in Google Colab directly from your existing `shuffled_v3` and `shuffled_v4` datasets:
1. **Prepares & Merges 13-KP Dataset**: Merges `shuffled_v3` + `shuffled_v4`, removes `ball_top` (5), `ankle_center` (12), and `shin_mid` (15), and verifies zero leakage.
2. **Calculates 13-KP Distance Priors**: Generates data-driven skeleton distance priors.
3. **Trains Fresh ARShoe M1v2**: 100 epochs with MobileNetV3-small backbone, 13 heatmap channels, 32 PAF channels.
4. **Evaluates Best Checkpoint**: Production skeleton-consistent decoding on validation split.
5. **Benchmarks Against 16-KP Baseline**: Automated side-by-side comparison table.

### Step 1: Verify Colab Environment & GPU

In [ ]:
!nvidia-smi
!pip install -q pyyaml tqdm albumentations opencv-python matplotlib

### Step 2: Set Working Directory & Pull Latest Code

In [ ]:
import os

# Ensure we are in shoes_vto repository directory
if os.path.exists('/content/shoes_vto_v2/shoes_vto'):
    %cd /content/shoes_vto_v2/shoes_vto
elif os.path.exists('/content/shoes_vto'):
    %cd /content/shoes_vto

# Pull latest 13-keypoint branch
!git checkout deletekp6and12
!git pull origin deletekp6and12

print('Current Working Directory:', os.getcwd())

### Step 3: Prepare 13-KP Dataset from Raw `shuffled_v3` & `shuffled_v4`
This one-line command:
- Reads `dataset/shuffled_v3` and `dataset/shuffled_v4`
- Performs SHA-256 deduplication and cross-split leakage checks
- Strips the 3 removed keypoints (`ball_top`, `ankle_center`, `shin_mid`)
- Writes clean 13-KP YOLO annotations to `dataset/merged_v3_v4_13kp`
- Generates `configs/skeleton_priors_train_13kp.json`

In [ ]:
!python tools/prepare_13kp_from_raw.py \
    --v3_dir dataset/shuffled_v3 \
    --v4_dir dataset/shuffled_v4 \
    --output_dir dataset/merged_v3_v4_13kp

### Step 4: Verify Dataset Integrity

In [ ]:
!python tools/verify_13kp_dataset.py --dataset_root dataset/merged_v3_v4_13kp

### Step 5: Launch 100-Epoch Fresh 13-KP Training
*(Takes ~10–12 minutes on Colab GPU with torch.compile enabled)*

In [ ]:
!python train_m1v2_13kp.py \
    --dataset_root dataset/merged_v3_v4_13kp \
    --output_dir outputs/m1v2_13kp_training \
    --epochs 100 \
    --batch_size 16 \
    --lr 0.0005

### Step 6: Evaluate Best Model on Validation Split

In [ ]:
!python evaluate_m1.py \
    --checkpoint outputs/m1v2_13kp_training/checkpoints/best.pth \
    --dataset_root dataset/merged_v3_v4_13kp \
    --split valid \
    --output_dir outputs/m1v2_13kp_valid_eval \
    --num_vis 10

### Step 7: Compare 13-KP Model Against 16-KP Baseline

In [ ]:
!python tools/compare_13kp_vs_16kp.py \
    --eval_13kp outputs/m1v2_13kp_valid_eval/evaluation_summary.json \
    --eval_16kp outputs/m1v2_valid_eval/evaluation_summary.json

### Step 8: Download Training & Evaluation Deliverables

In [ ]:
!zip -r m1v2_13kp_training_results.zip outputs/m1v2_13kp_training/checkpoints/best.pth outputs/m1v2_13kp_valid_eval/

from google.colab import files
files.download('m1v2_13kp_training_results.zip')